<hr>
<div>
<h2>Initialize Agent Environment</h2>
<p>Initializes the shared imports and data access used throughout section 05. The graph itself lives in <code>retention.agent</code>, so the nightly run (section 06) executes exactly what this section demonstrates. Every language-model response is cached in <code>cache/llm_cache.sqlite</code>: the first run calls gpt-4o-mini, and reruns replay the same responses at no cost.</p>
</div>

In [1]:
# Standard Library
import json
import os
import sys
import time
import warnings
from pathlib import Path

# See section 03: the shared environment's TensorFlow can't load, and sentence-transformers probes for it.
sys.modules['tensorflow'] = None
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from langchain_core.callbacks import UsageMetadataCallbackHandler
from langchain_core.globals import set_llm_cache
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.types import Command
from IPython.display import display

sys.path.insert(0, 'src')
from retention import agent as A, data, policy as P

warnings.filterwarnings('ignore')
pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 40)
pd.set_option('display.max_colwidth', 200)

In [2]:
# Plotting Theme and Color Palette
SURFACE = '#ffffff'
INK = '#0b0b0b'
MUTED = '#898781'
GRID = '#e1e0d9'
BASELINE = '#c3c2b7'
ACCENT = '#2a78d6'
ACCENT2 = '#eb6834'

plt.rcParams.update({
    'figure.dpi': 110,
    'figure.facecolor': SURFACE,
    'axes.facecolor': SURFACE,
    'savefig.facecolor': SURFACE,
    'axes.edgecolor': BASELINE,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'font.size': 9,
    'text.color': INK,
    'axes.labelcolor': INK,
    'xtick.color': MUTED,
    'ytick.color': MUTED,
})

In [3]:
# Output Helpers & Constants
OUTPUTS = Path('outputs')
CACHE = Path('cache')
SEED = 20260921
N_MONTHS = 24
BUDGET = 40_000
MODEL = 'gpt-4o-mini'
PRICE_PER_MILLION = {'input': 0.15, 'output': 0.60}   # gpt-4o-mini list prices, USD
RUN_ID = 'night-24'
CHECKPOINTS = OUTPUTS / 'agent_checkpoints.sqlite'
OUTBOX = OUTPUTS / 'outbox.jsonl'
RUN_LOG = CACHE / 'agent_run_log.json'
OFFER_LABELS = {'discount': 'Discount', 'device': 'Device credit', 'data': 'Data upgrade'}

if not os.environ.get('OPENAI_API_KEY'):
    env_file = Path('../../selected-work/.env')
    if env_file.exists():
        os.environ['OPENAI_API_KEY'] = next((line.split('=', 1)[1].strip().strip('"\'') for line in env_file.read_text().splitlines()
                                             if line.strip().startswith('OPENAI_API_KEY')), '')
os.environ.setdefault('OPENAI_API_KEY', 'cached-responses-only')
set_llm_cache(A.thread_safe_cache(CACHE / 'llm_cache.sqlite'))

<hr style="border: none; border-top: 2px solid #4A4A4A;">
<div>
<h1>05 · Retention Agent</h1>
<p><strong>Purpose:</strong> Turns tonight's plan into messages a person can approve. For each chosen customer, the agent reads their care notes and the offer policy, drafts an SMS, checks it against every hard rule and a policy review, and revises what fails. Customers with an open support ticket are routed to care instead. Nothing is sent until a reviewer approves.</p>
<p><strong>Architecture:</strong> A LangGraph workflow. The language model only reads and writes: who gets which offer, how each customer is routed, and every number come from the optimizer and the policy's rules in code, through tools. An earlier version let the model decide routing from the notes; section 4 shows why it no longer does.</p>
<table style="width: 100%; border-collapse: collapse;">
<thead><tr><th style="text-align: left;">Node</th><th style="text-align: left;">What it does</th><th style="text-align: left;">Built with</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;"><code>load_plan</code></td><td style="text-align: left; vertical-align: top;">Gets tonight's customers and offers from the optimizer</td><td style="text-align: left; vertical-align: top;">LangChain tool wrapping section 04</td></tr>
<tr><td style="text-align: left; vertical-align: top;"><code>customer</code> ×N</td><td style="text-align: left; vertical-align: top;">One branch per customer, run in parallel: gather context and route, then draft, check, and revise at most twice</td><td style="text-align: left; vertical-align: top;">LangGraph subgraph, fanned out with <code>Send</code></td></tr>
<tr><td style="text-align: left; vertical-align: top;">&nbsp;&nbsp;↳ gather context</td><td style="text-align: left; vertical-align: top;">The customer's care notes, the relevant policy sections, and the route: an open support ticket goes to care follow-up with no draft</td><td style="text-align: left; vertical-align: top;">LangChain retrievers over Chroma; routing tool in code</td></tr>
<tr><td style="text-align: left; vertical-align: top;">&nbsp;&nbsp;↳ draft</td><td style="text-align: left; vertical-align: top;">The SMS and a rationale for the reviewer</td><td style="text-align: left; vertical-align: top;">gpt-4o-mini, structured output</td></tr>
<tr><td style="text-align: left; vertical-align: top;">&nbsp;&nbsp;↳ check</td><td style="text-align: left; vertical-align: top;">Hard rules in code, then a policy review scored 1–5 on four criteria</td><td style="text-align: left; vertical-align: top;">LangChain tool; gpt-4o-mini as judge, structured output</td></tr>
<tr><td style="text-align: left; vertical-align: top;"><code>aggregate</code></td><td style="text-align: left; vertical-align: top;">Counts what passed, what was revised, and what was routed</td><td style="text-align: left; vertical-align: top;">Plain code</td></tr>
<tr><td style="text-align: left; vertical-align: top;"><code>human_approval</code></td><td style="text-align: left; vertical-align: top;">Pauses the run until a reviewer approves, edits, or rejects each draft</td><td style="text-align: left; vertical-align: top;">LangGraph <code>interrupt</code>, SQLite checkpointer</td></tr>
<tr><td style="text-align: left; vertical-align: top;"><code>dispatch</code></td><td style="text-align: left; vertical-align: top;">Re-checks every approved message, edits included, and writes it to a simulated outbox</td><td style="text-align: left; vertical-align: top;">Plain code</td></tr>
<tr><td style="text-align: left; vertical-align: top;"><code>report</code></td><td style="text-align: left; vertical-align: top;">The run's summary</td><td style="text-align: left; vertical-align: top;">Plain code</td></tr>
</tbody>
</table>
</div>

<hr>
<div>
<h2>1 · Tools and Retrieval</h2>
<p><strong>Purpose:</strong> Builds what the agent can use: vector stores over the care notes and the offer policy, and four LangChain tools. The tools are the agent's only source of facts.</p>
<p><strong>Design Notes:</strong></p>
<ol style="margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>Stores.</strong> Chroma collections embedded with section 03's local sentence model: the 6,000 care notes, each tagged with its account, month, and topic category; and the policy, one document per section.</li>
<li style="margin-bottom: 0.7em;"><strong>Tools.</strong> <code>plan_tonight</code> (the optimizer's customers and numbers), <code>retrieve_notes</code> (a customer's notes, filtered to their account), <code>retrieve_policy</code> (the sections relevant to an offer), <code>route_customer</code> (the policy's escalation rule: an open support ticket goes to care), and <code>check_rules</code> (every hard rule, including eligibility tonight).</li>
<li style="margin-bottom: 0.7em;"><strong>Tonight's Batch.</strong> 40 of the plan's 614 contacted customers: 3 with an open support ticket, to exercise routing; 27 more with care notes; and 10 without notes, at a demonstration's cost.</li>
</ol>
<p style="margin-bottom: 0;"><strong>Observations:</strong></p>
<ol style="margin-top: 0; margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>Retrieval stays on the customer.</strong> Notes are filtered to the customer's own account before ranking, so one customer's words can never reach another's message. The policy search returns the message standards, contact rules, escalation, and the offer's eligibility for a discount.</li>
</ol>
</div>

In [4]:
# Build Stores and Tools
snapshot = data.load('scoring_snapshot')
notes = data.load('care_notes')
note_topics = pd.read_parquet(OUTPUTS / 'note_topics.parquet')
plan = pd.read_parquet(OUTPUTS / 'tonight_plan.parquet')
contacted = plan[~plan['holdout']]

embeddings = A.SentenceEmbeddings()
note_store, policy_store = A.build_stores(notes, note_topics, data.load_policy(), embeddings)

open_ticket = contacted['account_id'].isin(snapshot.loc[snapshot['unresolved_ticket'], 'account_id'])
ticketed = contacted[open_ticket].sample(3, random_state = SEED)
with_notes = contacted[~open_ticket & contacted['account_id'].isin(notes['account_id'])].sample(27, random_state = SEED)
without_notes = contacted[~open_ticket & ~contacted['account_id'].isin(notes['account_id'])].sample(10, random_state = SEED)
batch = pd.concat([ticketed, with_notes, without_notes])


def as_customer(row):
    return {'account_id': int(row.account_id), 'offer': row.offer, 'offer_name': OFFER_LABELS[row.offer].lower(),
            'offer_terms': P.OFFER_TERMS[row.offer],
            'why': (f'estimated 90-day churn risk {row.p_churn_90:.1%}; this offer is estimated to lower it by {row.uplift * 100:.1f} points; '
                    f'rep codes point to {row.reason or "no clear reason"}; 24-month value about ${row.value:,.0f}')}


customers = [as_customer(r) for r in batch.itertuples()]
plan_summary = {'plan_customers': int(len(contacted)), 'plan_held_out': int(plan['holdout'].sum()), 'batch': len(customers),
                'exposure': float(contacted['exposure'].sum())}
tools = A.make_tools(customers, plan_summary, note_store, policy_store, snapshot, N_MONTHS)

example = customers[0]
print(f"{len(notes):,} notes and {len(policy_store.get()['ids'])} policy sections indexed · batch of {len(customers)} customers\n")
print('retrieve_notes →', json.dumps(tools['retrieve_notes'].invoke({'account_id': example['account_id']}), indent = 1)[:700])
print('\nretrieve_policy →', [doc.splitlines()[0] for doc in tools['retrieve_policy'].invoke({'offer': example['offer']})])
print('route_customer →', tools['route_customer'].invoke({'account_id': example['account_id']}))

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

6,000 notes and 7 policy sections indexed · batch of 40 customers

retrieve_notes → []

retrieve_policy → ['## Message standards', '## Escalation (hard)', '## Contact rules (hard)', '## Eligibility: loyalty discount (hard)']
route_customer → care_follow_up


<hr>
<div>
<h2>2 · Tonight's Run</h2>
<p><strong>Purpose:</strong> Runs the graph for tonight's batch, up to the point where it pauses for approval.</p>
<p><strong>Design Notes:</strong></p>
<ol style="margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>Models.</strong> gpt-4o-mini at temperature 0 for both drafting and review.</li>
<li style="margin-bottom: 0.7em;"><strong>Checkpointing.</strong> Every step is saved to a SQLite database under a run ID. The pause for approval can last hours; any process can resume the run from where it stopped (section 5).</li>
<li style="margin-bottom: 0.7em;"><strong>Usage.</strong> Tokens and time are recorded on the first, uncached run and shown from that record afterward, so a rerun from the cache still reports the real cost.</li>
</ol>
<p style="margin-bottom: 0;"><strong>Observations:</strong></p>
<ol style="margin-top: 0; margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>40 customers in 14 seconds, for about a cent.</strong> 52,410 input and 4,789 output tokens: &#36;0.011, or &#36;0.27 per 1,000 customers.</li>
<li style="margin-bottom: 0.7em;"><strong>Almost everything passes on the first draft.</strong> 37 offers drafted, 36 passing on the first try and all 37 after one revision; 3 customers routed to care by the ticket rule, with no draft at all.</li>
</ol>
</div>

In [5]:
# Run Tonight's Batch
CHECKPOINTS.unlink(missing_ok = True)
drafter = ChatOpenAI(model = MODEL, temperature = 0)
judge = ChatOpenAI(model = MODEL, temperature = 0)
usage = UsageMetadataCallbackHandler()
config = {'configurable': {'thread_id': RUN_ID}, 'callbacks': [usage], 'max_concurrency': 8}

with SqliteSaver.from_conn_string(str(CHECKPOINTS)) as checkpointer:
    graph = A.build_graph(drafter, judge, tools, checkpointer, OUTBOX)
    started = time.time()
    graph.invoke({'month': N_MONTHS, 'budget': float(BUDGET)}, config)
    seconds = time.time() - started
    paused = graph.get_state(config)

tokens = {'input': sum(u['input_tokens'] for u in usage.usage_metadata.values()),
          'output': sum(u['output_tokens'] for u in usage.usage_metadata.values())}
if not RUN_LOG.exists():   # the first run is the live one; cached responses report their original usage but take no time
    RUN_LOG.write_text(json.dumps({'seconds': seconds, 'tokens': tokens, 'customers': len(customers)}, indent = 1))
log = json.loads(RUN_LOG.read_text())
cost = sum(log['tokens'][k] * PRICE_PER_MILLION[k] / 1e6 for k in PRICE_PER_MILLION)
print(f"paused before: {paused.next} · this run {seconds:.1f} s · first run {log['seconds']:.0f} s, "
      f"{log['tokens']['input']:,} input and {log['tokens']['output']:,} output tokens, ${cost:.3f} "
      f"(${cost / log['customers'] * 1000:.2f} per 1,000 customers)")
display(pd.Series(paused.values['review']).to_frame('tonight'))

paused before: ('human_approval',) · this run 0.6 s · first run 14 s, 52,410 input and 4,789 output tokens, $0.011 ($0.27 per 1,000 customers)


,tonight
drafts,40
offers_drafted,37
passed,37
passed_first_try,36
care_follow_up,3


<hr>
<div>
<h2>3 · The Drafts</h2>
<p><strong>Purpose:</strong> Shows what the agent proposes: the action for each customer, the message, the reviewer's rationale, and how the checks went.</p>
<p style="margin-bottom: 0;"><strong>Observations:</strong></p>
<ol style="margin-top: 0; margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>The messages are compliant and plain.</strong> About 229 characters, the offer's exact terms, a thank-you, and the opt-out line. Where a customer's notes show they asked about their bill, the message says so briefly (“to help with your bill”).</li>
<li style="margin-bottom: 0.7em;"><strong>They are also similar to one another.</strong> Most open with the same appreciation line. That's acceptable for a first release; a reviewer who wants variety would add style guidance, and the review would still hold every version to the same rules.</li>
</ol>
</div>

In [6]:
# Review the Drafts
drafts = pd.DataFrame(paused.values['drafts'])
for column in ['terms_accurate', 'friendly_and_direct', 'no_pressure', 'respects_privacy']:
    drafts[column] = drafts['judgment'].map(lambda j: j.get(column) if j else np.nan)
drafts['characters'] = drafts['message'].str.len()

display(drafts.groupby('action').agg(customers = ('account_id', 'size'), passed = ('passed', 'mean'), revisions = ('attempts', lambda a: (a - 1).clip(lower = 0).sum()),
                                     mean_characters = ('characters', 'mean')).style.format({'passed': '{:.0%}', 'mean_characters': '{:.0f}'}))
for d in drafts.sort_values('attempts', ascending = False).head(6).itertuples():
    print(f"[{d.account_id} · {d.action} · {d.attempts} draft{'s' if d.attempts > 1 else ''} · {'passed' if d.passed else 'FAILED'}]")
    print(f"  message:   {d.message or '—'}")
    print(f"  rationale: {d.rationale}\n")

,customers,passed,revisions,mean_characters
action,,,,
care_follow_up,3,100%,0,0
send_offer,37,100%,1,229


[13913 · send_offer · 2 drafts · passed]
  message:   Hi there! We have a special offer for you: enjoy $10 off your monthly bill for the next 6 months. If you have any questions or need assistance, feel free to reach out! Reply STOP to opt out.
  rationale: This offer provides a direct benefit to the customer by addressing their concern about the monthly bill without implying pressure. The terms are clearly stated, aligning with the policy standards.

[16809 · send_offer · 1 draft · passed]
  message:   Hi there! We appreciate your loyalty. To help with your bill, we're offering you $10 off your monthly bill for the next 6 months. We hope this makes your experience even better! Let us know if you have any questions. Reply STOP to opt out.
  rationale: The customer expressed dissatisfaction with their bill, and this offer directly addresses their concerns by providing a discount. They are eligible for the offer based on their account history.

[43638 · send_offer · 1 draft · passed]
  m

<hr>
<div>
<h2>4 · The Checks</h2>
<p><strong>Purpose:</strong> Measures how well the drafts meet the policy, and tests the checks themselves on messages written to break the rules.</p>
<p><strong>Design Notes:</strong></p>
<ol style="margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>Two Layers.</strong> Code checks every hard rule: 320 characters, the opt-out line, the offer's exact terms, no mention of churn or risk, no pressure words, no competitor, and eligibility tonight. The language-model review scores what code can't: accuracy, tone, pressure, and privacy. A draft passes only if code finds nothing and every score is 4 or 5.</li>
<li style="margin-bottom: 0.7em;"><strong>Stress Test.</strong> Six messages, each written to break one rule, and one compliant baseline go through both layers. A check that can't catch a planted violation can't be trusted on a real one.</li>
<li style="margin-bottom: 0.7em;"><strong>Routing.</strong> Each customer's route against their hidden reason from the answer key. Care follow-ups should be service problems, and offers shouldn't be withheld from the price-sensitive customers the discount is for.</li>
</ol>
<p style="margin-bottom: 0;"><strong>Observations:</strong></p>
<ol style="margin-top: 0; margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>The drafts meet the policy.</strong> 97% pass on the first draft and 100% in the end; the lowest review score given is 4 out of 5.</li>
<li style="margin-bottom: 0.7em;"><strong>Both layers are needed.</strong> Every planted violation is caught and the compliant baseline passes. Code catches five of the six outright; the invented detail (“since you've been streaming more video lately”) breaks no written rule, and only the review catches it.</li>
<li style="margin-bottom: 0.7em;"><strong>Routing now keeps the discount where it belongs.</strong> 32 of the batch's 34 price-sensitive customers get their offer. The ticket rule sends 3 customers to care, 2 of them price-sensitive: the policy's choice, made visible.</li>
</ol>
</div>

In [7]:
# Score the Checks
offers = drafts[drafts['action'] == 'send_offer']
scores = offers[['terms_accurate', 'friendly_and_direct', 'no_pressure', 'respects_privacy']]
display(pd.DataFrame({
    'offers drafted': [len(offers)], 'passed first draft': [(offers['attempts'] == 1).mean()], 'passed in the end': [offers['passed'].mean()],
    'mean review score': [scores.mean().mean()], 'lowest score given': [scores.min().min()],
}).style.format({'passed first draft': '{:.0%}', 'passed in the end': '{:.0%}', 'mean review score': '{:.2f}', 'lowest score given': '{:.0f}'}))

ok = example['account_id']
planted = pd.DataFrame([
    ('baseline, compliant', f"Hi there! You can get {P.OFFER_TERMS['discount']}. Reply YES and we'll add it to your account. {P.OPT_OUT}"),
    ('too long', 'Hi there! ' + 'We value you as a customer. ' * 12 + f"Get {P.OFFER_TERMS['discount']}. {P.OPT_OUT}"),
    ('no opt-out', f"Hi there! You can get {P.OFFER_TERMS['discount']}. Reply YES to add it."),
    ('wrong terms', f"Hi there! You can get $15 off your monthly bill for 3 months. {P.OPT_OUT}"),
    ('pressure', f"Hi there! Act now: {P.OFFER_TERMS['discount']}, today only. {P.OPT_OUT}"),
    ('reveals the prediction', f"Hi there! We noticed you might leave, so here's {P.OFFER_TERMS['discount']}. {P.OPT_OUT}"),
    ('invents a detail', f"Hi there! Since you've been streaming more video lately, enjoy {P.OFFER_TERMS['discount']}. {P.OPT_OUT}"),
], columns = ['planted problem', 'message'])
judge_structured = judge.with_structured_output(A.Judgment)
policy_excerpts = tools['retrieve_policy'].invoke({'offer': 'discount'})


def review(message):
    return judge_structured.invoke([('system', A.JUDGE_SYSTEM), ('user', f"Offer, exactly: {P.OFFER_TERMS['discount']}.\nPolicy excerpts:\n"
                                     + '\n\n'.join(policy_excerpts) + f"\n\nWhat the customer told us (care notes):\n- nothing\n\nMessage to review:\n{message}")])


planted['code finds'] = planted['message'].map(lambda m: '; '.join(tools['check_rules'].invoke({'message': m, 'offer': 'discount', 'account_id': ok})) or '—')
reviews = planted['message'].map(review)
planted['lowest review score'] = reviews.map(lambda r: min(v for k, v in r.model_dump().items() if k != 'issues'))
planted['caught'] = (planted['code finds'] != '—') | (planted['lowest review score'] < 4)
display(planted.drop(columns = 'message'))

,offers drafted,passed first draft,passed in the end,mean review score,lowest score given
0,37,97%,100%,4.92,4


,planted problem,code finds,lowest review score,caught
0,"baseline, compliant",—,5,False
1,too long,SMS is 411 characters; the limit is 320,2,True
2,no opt-out,"SMS must end with ""Reply STOP to opt out.""",4,True
3,wrong terms,"offer terms missing: $10, 6 months",1,True
4,pressure,pressure or deadline,1,True
5,reveals the prediction,mentions churn or prediction,1,True
6,invents a detail,—,1,True


In [8]:
# Check Routing against the Answer Key
truth_accounts = data.load_answer_key('truth_accounts').set_index('account_id')
routing = drafts.assign(hidden_reason = drafts['account_id'].map(truth_accounts['reason']),
                        has_notes = drafts['account_id'].isin(notes['account_id']))
display(pd.crosstab(routing['action'], routing['hidden_reason']).rename_axis(index = 'agent\'s action', columns = 'hidden reason (answer key)'))

hidden reason (answer key),device,network,price,service
agent's action,,,,
care_follow_up,1,0,2,0
send_offer,2,2,32,1


<hr>
<div>
<h2>5 · Human Approval</h2>
<p><strong>Purpose:</strong> Completes the run the way a reviewer would: from a fresh process, it reloads the paused run from its checkpoint, approves, edits, and rejects drafts, and lets the graph finish.</p>
<p><strong>Design Notes:</strong></p>
<ol style="margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>Resuming.</strong> A new graph object and a new database connection pick up the run by its ID, the way a review screen would hours later.</li>
<li style="margin-bottom: 0.7em;"><strong>Tonight's Decisions.</strong> The reviewer approves every offer that passed, except one they reject; edits one message to be warmer; and makes one careless edit that drops the opt-out line. Drafts that failed the checks, and customers routed to care, aren't approved.</li>
<li style="margin-bottom: 0.7em;"><strong>Last Line of Defense.</strong> <code>dispatch</code> re-checks every approved message, edits included, and holds back any that break a hard rule.</li>
</ol>
<p style="margin-bottom: 0;"><strong>Observations:</strong></p>
<ol style="margin-top: 0; margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>The run survived the pause.</strong> A new graph and a new connection reloaded the run at <code>human_approval</code> with all 40 drafts and finished it.</li>
<li style="margin-bottom: 0.7em;"><strong>The reviewer stays in control, and so do the rules.</strong> 36 offers approved (one edited), one rejected, and 35 messages sent: the careless edit that dropped the opt-out line was held back by <code>dispatch</code>, and the rejected draft never reached the outbox.</li>
</ol>
</div>

In [9]:
# Approve and Finish the Run
passed_offers = [int(a) for a in offers.loc[offers['passed'], 'account_id']]
rejected, warmer, careless = passed_offers[0], passed_offers[1], passed_offers[2]
decisions = {
    'approve': [a for a in passed_offers if a not in (rejected, warmer, careless)],
    'reject': [rejected],
    'edits': {warmer: f"Hi there! Thanks for being with us. As a small thank-you, you can get {P.OFFER_TERMS['discount']}. Reply YES to add it. {P.OPT_OUT}",
              careless: f"Hi there! You can get {P.OFFER_TERMS['discount']}. Reply YES to add it."},
}

with SqliteSaver.from_conn_string(str(CHECKPOINTS)) as checkpointer:
    reviewer_graph = A.build_graph(drafter, judge, tools, checkpointer, OUTBOX)
    resumed = reviewer_graph.get_state({'configurable': {'thread_id': RUN_ID}})
    print(f'reloaded run {RUN_ID!r}: waiting at {resumed.next}, {len(resumed.values["drafts"])} drafts')
    final = reviewer_graph.invoke(Command(resume = decisions), {'configurable': {'thread_id': RUN_ID}})

display(pd.Series(final['report']).to_frame('run report'))
outbox = pd.read_json(OUTBOX, lines = True)
print(f"{len(outbox)} messages in the outbox · careless edit sent: {careless in outbox['account_id'].values} · "
      f"rejected sent: {rejected in outbox['account_id'].values}")
display(outbox.head(5))

reloaded run 'night-24': waiting at ('human_approval',), 40 drafts


,run report
drafts,40
offers_drafted,37
passed,37
passed_first_try,36
care_follow_up,3
approved,36
sent,35
edited,1
rejected,1
plan,"{'plan_customers': 614, 'plan_held_out': 52, 'batch': 40, 'exposure': 36840.0}"


35 messages in the outbox · careless edit sent: False · rejected sent: False


,account_id,offer,channel,message,edited
0,50988,discount,sms,"Hi there! Thanks for being with us. As a small thank-you, you can get $10 off the monthly bill for 6 months. Reply YES to add it. Reply STOP to opt out.",True
1,57070,discount,sms,"Hi there! We appreciate you being with us. To help with your bill, we're offering you $10 off your monthly bill for the next 6 months. If you have any questions or want to discuss this offer, just...",False
2,95948,discount,sms,"Hi there! We appreciate you being with us. To help with your bill, we're offering you $10 off your monthly bill for the next 6 months. We hope this helps make your experience even better! Reply ST...",False
3,75732,discount,sms,Hi there! We appreciate your loyalty. We're offering you $10 off your monthly bill for the next 6 months. This is our way of saying thank you for being with us. If you have any questions or want t...,False
4,49479,discount,sms,"Hi there! We appreciate your loyalty. To help with your bill, we're offering you $10 off your monthly bill for the next 6 months. If you have any questions or want to discuss your plan options, ju...",False


<hr>
<div>
<h2>6 · Cost and Speed</h2>
<p><strong>Purpose:</strong> Puts the first run's measured tokens and time in terms of a full night.</p>
<p style="margin-bottom: 0;"><strong>Observations:</strong></p>
<ol style="margin-top: 0; margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>A full night costs about 16 cents.</strong> Tonight's 614 customers would take about 4 minutes at 8 drafts in parallel; even 10,000 customers would cost under &#36;3. The language model is not the expensive part of this program; the offers are.</li>
</ol>
</div>

In [10]:
# Scale Cost and Time to a Full Night
per_customer_cost = cost / log['customers']
per_customer_seconds = log['seconds'] / log['customers']
nightly = pd.DataFrame({'customers': [40, 614, 2_000, 10_000]})
nightly['cost, $'] = nightly['customers'] * per_customer_cost
nightly['minutes at 8 in parallel'] = nightly['customers'] * per_customer_seconds / 60
display(nightly.style.format({'customers': '{:,}', 'cost, $': '{:,.2f}', 'minutes at 8 in parallel': '{:,.1f}'})
        .set_caption('Scaled linearly from the measured 40-customer run'))

,customers,"cost, $",minutes at 8 in parallel
0,40,0.01,0.2
1,614,0.16,3.7
2,"2,000",0.54,11.9
3,"10,000",2.68,59.7


<hr>
<div>
<h2>7 · What Changed While Building It</h2>
<p><strong>Purpose:</strong> Records three problems the first versions of the agent had, how each was found, and what fixed it. All three point the same way: rules and fixed text belong in code, and the language model should only do what code can't.</p>
<table style="width: 100%; border-collapse: collapse;">
<thead><tr><th style="text-align: left;">First version</th><th style="text-align: left;">What went wrong</th><th style="text-align: left;">Fix</th><th style="text-align: left;">Result</th></tr></thead>
<tbody>
<tr><td style="text-align: left; vertical-align: top;">The model chose each customer's route from the notes</td><td style="text-align: left; vertical-align: top;">It sent 16 of 40 customers to care follow-up; the answer key showed 13 of them were price-sensitive, the customers the discount is for. A quarter of notes are about some other issue (section 03), and the model read any complaint as a service problem</td><td style="text-align: left; vertical-align: top;">Routing moved to code, from the ticket system: an open support ticket goes to care</td><td style="text-align: left; vertical-align: top;">3 routed; 32 of 34 price-sensitive customers get their offer</td></tr>
<tr><td style="text-align: left; vertical-align: top;">Drafter and reviewer had different instructions</td><td style="text-align: left; vertical-align: top;">The drafter was allowed to acknowledge what a customer said; the reviewer penalized it, and even “we appreciate your loyalty”, so drafts churned through revisions and two never passed</td><td style="text-align: left; vertical-align: top;">Both read the same rule: thanking and briefly acknowledging what the customer told us are fine, inferring is not</td><td style="text-align: left; vertical-align: top;">Every draft passes in the end</td></tr>
<tr><td style="text-align: left; vertical-align: top;">The model wrote the opt-out line</td><td style="text-align: left; vertical-align: top;">All 20 first-draft failures in one run were the same: the model left off “Reply STOP to opt out.”</td><td style="text-align: left; vertical-align: top;">Code appends the line; the model writes only the body</td><td style="text-align: left; vertical-align: top;">First-draft pass rate from 46% to 97%; a third less cost per customer</td></tr>
</tbody>
</table>
</div>

<hr>
<div>
<h2>Section 05 Summary</h2>
<ol style="margin-left: 0; padding-left: 1.25em;">
<li style="margin-bottom: 0.7em;"><strong>A working agent, end to end.</strong> A LangGraph workflow fans out one branch per customer, retrieves their notes and the policy through LangChain tools and retrievers, drafts with structured output, checks every hard rule in code and the rest with a language-model review, revises what fails, pauses for a person's approval, survives the pause through a checkpoint, and re-checks everything before sending.</li>
<li style="margin-bottom: 0.7em;"><strong>It's cheap and fast.</strong> About &#36;0.27 per 1,000 customers, and 4 minutes for a full night.</li>
<li style="margin-bottom: 0.7em;"><strong>The design rule held.</strong> Each problem found while building it was solved by moving a decision or a fixed piece of text out of the model and into code. What's left to the model is what it's good at: reading a customer's notes and writing a plain, accurate message.</li>
<li style="margin-bottom: 0.7em;"><strong>Next.</strong> Section 06 runs it nightly, watches for drift, and measures the plan's real effect against the held-out customers.</li>
</ol>
</div>